Step 1: Read Data

In [0]:
from pyspark.sql.functions import *

In [0]:
data_frame = spark.read.csv("/Volumes/dev/demo/raw/sales.csv", header=True, inferSchema=True)

In [0]:
data_frame.display()
data_frame.printSchema()
data_frame.count()

Step 2: Explore the Data

In [0]:
data_frame.count()

In [0]:
print(len(data_frame.columns))

In [0]:
data_frame.select("customer_id").distinct().display()

In [0]:
data_frame.select("product_id").distinct().display()

In [0]:
data_frame.summary()

Step 3: Clean the Data

In [0]:
newDf = data_frame.dropDuplicates()

In [0]:
newDf = newDf.dropDuplicates(['transaction_id']) 

In [0]:
newDf = newDf.dropna()

Step 4: Transform the Data

In [0]:
newDf = newDf.withColumnRenamed("order_id", "orderId")

In [0]:

newDf = newDf.withColumn("net_amount", round(col("total_amount") - col("discount_amount"), 2))

In [0]:
newDf = newDf.withColumn("order_date", to_date(col("order_date"), format='dd-MM-yyyy')) 

Step 5: Write the Data

In [0]:
newDf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.demo.orders_clean")

Step 6: Analytics

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_total_sales AS
SELECT
    ROUND(SUM(net_amount), 2) AS total_sales
FROM dev.demo.orders_clean;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_total_orders AS
SELECT
    COUNT(DISTINCT orderId) AS total_orders
FROM dev.demo.orders_clean;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_unique_customers AS
SELECT
    COUNT(DISTINCT customer_id) AS unique_customers
FROM dev.demo.orders_clean;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_customer_sales AS
SELECT
    customer_id,
    ROUND(SUM(net_amount), 2) AS total_sales
FROM dev.demo.orders_clean
GROUP BY customer_id
ORDER BY total_sales DESC;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_product_sales AS
SELECT
    product_id,
    ROUND(SUM(net_amount), 2) AS total_sales
FROM dev.demo.orders_clean
GROUP BY product_id
ORDER BY total_sales DESC;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_top_10_customers AS
SELECT
    customer_id,
    ROUND(SUM(net_amount), 2) AS total_sales
FROM dev.demo.orders_clean
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_top_10_products AS
SELECT
    product_id,
    ROUND(SUM(net_amount), 2) AS total_sales
FROM dev.demo.orders_clean
GROUP BY product_id
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_highest_sales_transaction AS
SELECT
    *
FROM dev.demo.orders_clean
ORDER BY net_amount DESC
LIMIT 1;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_lowest_sales_transaction AS
SELECT
    *
FROM dev.demo.orders_clean
ORDER BY net_amount ASC
LIMIT 1;

In [0]:
%sql

CREATE OR REPLACE VIEW dev.demo.v_average_order_value AS
SELECT
    ROUND(AVG(order_total), 2) AS average_order_value
FROM (
    SELECT
        orderId,
        SUM(net_amount) AS order_total
    FROM dev.demo.orders_clean
    GROUP BY orderId
);